# 13 · Memory on the CLI boundary + per-term model  (~2–3 h)

> Part of the v0.2.0 experiments (notebooks 10–19), run with the fastapriori 0.2.0 build and the `bench/exp_runners.py` harness. Defaults: 1 warm-up + 3 measured runs per cell, full repetitions even for slow cells, explicit thread counts, failures recorded. Never run two of these notebooks at the same time.

Feeds §6.5 (Fig. 4) and the memory model in §3.3.

* `hybcli_*` run the Rust kernel in a numpy-only process (no pandas, no fastapriori import),
  reading pre-encoded arrays: the same boundary as the Borgelt/LCM command-line runs.
* `hybcli_load` is the same process without the call: interpreter + input baseline.
* Report both raw peak RSS and peak − load-only.
* The per-term model (index, anchor TID cache, thread buffers) is computed below from
  the data and the mined (k−1) level.

Output: `results/runs/<campaign>/memory_boundary.csv`, `results/runs/<campaign>/memory_terms.csv`.

In [1]:
import os, sys, json, math, time
from pathlib import Path
BUNDLE = Path.cwd().resolve().parent     # notebooks run from <bundle>/notebooks
assert (BUNDLE / "bench" / "exp_runners.py").exists(), "start Jupyter in <bundle>/notebooks"
sys.path.insert(0, str(BUNDLE / "bench"))
# Results of these notebooks go to results/runs/<FA_CAMPAIGN>/ (set it before starting Jupyter
# to keep campaigns apart; every run_grid call also appends to manifest.jsonl there).
os.environ.setdefault("FA_CAMPAIGN", "new_campaign")
import numpy as np, pandas as pd
import exp_runners as pr
import exp_data as pdd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
print("campaign", pr.CAMPAIGN, "->", pr.RESULTS)
print("build", pr.build_id(), "| extension", pr.ext_path())

campaign vm_20261029 -> /home/user/artifacts/results/runs/vm_20261029
build cc508b6408f4 | extension /home/user/.venvs/fastapriori/lib/python3.12/site-packages/fastapriori/_fastapriori_rs.cpython-312-x86_64-linux-gnu.so


In [2]:
DATASETS = ["Chainstore", "Kosarak", "Instacart", "T20.I4.D1000k.N50k.L50k",
            "T30.I8.D2000k.N10k.L10k", "T25.I8.D5000k.N40k.L40k"]
S = 1e-4
K2 = ["hybcli_load", "hybcli_pairs_v0", "hybcli_pairs_v2", "hybcli_pairs_v0_1t", "hybcli_pairs_v2_1t",
      "borgelt_apriori", "borgelt_eclat", "borgelt_fpgrowth", "lcm"]
K3 = ["hybcli_load", "hybcli_anchor", "hybcli_seeded", "borgelt_eclat", "borgelt_fpgrowth", "lcm"]

In [3]:
pr.run_grid("memory_boundary", DATASETS, K2, ks=[2], supports=[S], timeout_s=1500)
pr.run_grid("memory_boundary", DATASETS, K3, ks=[3], supports=[S], timeout_s=1500)
s = pr.summarize(pr.load_results("memory_boundary"))
rss = s.pivot_table(index=["dataset", "k"], columns="method", values="peak_rss_kb") / 1024
base = rss.pop("hybcli_load") if "hybcli_load" in rss else 0
delta = rss.filter(like="hybcli_").sub(base, axis=0).add_suffix("_minus_load")
display(rss.round(0)); display(delta.round(0))

preflight ok: 9 methods, build cc508b6408f4


memory_boundary:   0%|          | 0/54 [00:00<?, ?cell/s]

[memory_boundary] Chainstore | hybcli_load | k=2 s=0.0001 run 0: ok wall=nan rss=122676.0 (4.5s)
[memory_boundary] Chainstore | hybcli_load | k=2 s=0.0001 run 1: ok wall=nan rss=122688.0 (0.2s)


[memory_boundary] Chainstore | hybcli_load | k=2 s=0.0001 run 2: ok wall=nan rss=122724.0 (0.2s)
[memory_boundary] Chainstore | hybcli_load | k=2 s=0.0001 run 3: ok wall=nan rss=122688.0 (0.2s)


[memory_boundary] Chainstore | hybcli_pairs_v0 | k=2 s=0.0001 run 0: ok wall=0.6530111140018562 rss=345148.0 (0.8s)


[memory_boundary] Chainstore | hybcli_pairs_v0 | k=2 s=0.0001 run 1: ok wall=0.6620558370013896 rss=345356.0 (0.8s)


... (344 outputs left out)

[memory_boundary] T25.I8.D5000k.N40k.L40k | lcm | k=3 s=0.0001 run 2: ok wall=74.5 rss=1350424.0 (74.5s)


[memory_boundary] T25.I8.D5000k.N40k.L40k | lcm | k=3 s=0.0001 run 3: ok wall=74.2 rss=1350368.0 (74.2s)


method                     borgelt_apriori  borgelt_eclat  borgelt_fpgrowth  hybcli_anchor  hybcli_pairs_v0  hybcli_pairs_v0_1t  hybcli_pairs_v2  hybcli_pairs_v2_1t  hybcli_seeded     lcm
dataset                 k                                                                                                                                                                  
Chainstore              2            402.0          193.0             330.0            NaN            337.0               337.0            337.0               337.0            NaN   138.0
                        3              NaN          179.0             324.0          564.0              NaN                 NaN              NaN                 NaN          369.0   138.0
Instacart               2            674.0          722.0            1718.0            NaN           1240.0              1240.0           1240.0              1240.0            NaN   462.0
                        3              NaN          684.0            1712.0         4714.0              NaN                 NaN              NaN                 NaN         1365.0   462.0
Kosarak                 2            182.0          176.0            2988.0            NaN            422.0               334.0            430.0               334.0            NaN   128.0
                        3              NaN          174.0            3017.0            NaN              NaN                 NaN              NaN                 NaN         3469.0   128.0
T20.I4.D1000k.N50k.L50k 2           3859.0          388.0             828.0            NaN            722.0               722.0            722.0               722.0            NaN   228.0
                        3              NaN          388.0             828.0          722.0              NaN                 NaN              NaN                 NaN          799.0   228.0
T25.I8.D5000k.N40k.L40k 2           4236.0         2274.0            5121.0            NaN           4569.0              4569.0           4569.0              4569.0            NaN  1319.0
                        3              NaN         2274.0            5121.0         4569.0              NaN                 NaN              NaN                 NaN         5046.0  1319.0
T30.I8.D2000k.N10k.L10k 2            790.0         1071.0            2621.0            NaN           2220.0              2220.0           2220.0              2220.0            NaN   605.0
                        3              NaN         1071.0            2621.0         5659.0              NaN                 NaN              NaN                 NaN         2449.0   605.0

method                     hybcli_anchor_minus_load  hybcli_pairs_v0_minus_load  hybcli_pairs_v0_1t_minus_load  hybcli_pairs_v2_minus_load  hybcli_pairs_v2_1t_minus_load  hybcli_seeded_minus_load
dataset                 k                                                                                                                                                                          
Chainstore              2                       NaN                       217.0                          217.0                       217.0                          217.0                       NaN
                        3                     444.0                         NaN                            NaN                         NaN                            NaN                     249.0
Instacart               2                       NaN                       841.0                          841.0                       841.0                          841.0                       NaN
                        3                    4315.0                         NaN                            NaN                         NaN                            NaN                     966.0
Kosarak                 2                       NaN                       302.0                          214.0                       310.0                          214.0                       NaN
                        3                       NaN                         NaN                            NaN                         NaN                            NaN                    3350.0
T20.I4.D1000k.N50k.L50k 2                       NaN                       465.0                          465.0                       465.0                          465.0                       NaN
                        3                     465.0                         NaN                            NaN                         NaN                            NaN                     542.0
T25.I8.D5000k.N40k.L40k 2                       NaN                      3111.0                         3110.0                      3111.0                         3110.0                       NaN
                        3                    3110.0                         NaN                            NaN                         NaN                            NaN                    3588.0
T30.I8.D2000k.N10k.L10k 2                       NaN                      1506.0                         1506.0                      1506.0                         1506.0                       NaN
                        3                    4944.0                         NaN                            NaN                         NaN                            NaN                    1735.0

## Per-term memory model
index ≈ 2 · rows · 4 B (CSR both ways) + offsets; anchor TID cache at level k = Σ_{σ∈F_{k−1}} c(σ) · 4 B
(upper bound: Roaring compresses large lists); thread buffers = threads · N · 8 B; dense
triangular array (Borgelt Apriori) = |F1|²/2 · 4 B.

In [4]:
from fastapriori import find_associations
THREADS = os.cpu_count()
terms = []
for d in DATASETS:
    df = pdd.load_df(d)
    st = pdd.cost_stats(df, (S,))[0]
    f2 = find_associations(df, transaction_col="txn_id", item_col="item", k=2, min_support=S,
                           min_confidence=0.0, sorted_by=None, low_memory=False)
    anchor_cache_k3 = float(f2["instances"].sum() / 2) * 4     # each pair appears twice
    terms.append(dict(dataset=d, rows=st["rows"], N=st["N"], F1=st["F1"],
                      index_MB=(2 * st["rows"] * 4 + (st["D"] + st["N"]) * 4) / 2**20,
                      anchor_cache_k3_MB=anchor_cache_k3 / 2**20,
                      thread_buffers_MB=THREADS * st["N"] * 8 / 2**20,
                      triangular_MB=st["F1"] ** 2 / 2 * 4 / 2**20,
                      output_k2_MB=f2.memory_usage(deep=True).sum() / 2**20))
    del df, f2
mt = pd.DataFrame(terms); mt.to_csv(pr.RESULTS / "memory_terms.csv", index=False); mt.round(1)

,dataset,rows,N,F1,index_MB,anchor_cache_k3_MB,thread_buffers_MB,triangular_MB,output_k2_MB
0,Chainstore,8042879,46086,12194,65.8,18.4,2.8,283.6,3.0
1,Kosarak,8018988,41270,6494,65.1,467.6,2.5,80.4,81.6
2,Instacart,32434489,49677,10906,259.9,197.0,3.0,226.9,8.7
3,T20.I4.D1000k.N50k.L50k,20004006,49046,43626,156.6,13.1,3.0,3630.1,3.9
4,T30.I8.D2000k.N10k.L10k,59990826,9995,9984,465.4,337.6,0.6,190.1,29.2
5,T25.I8.D5000k.N40k.L40k,124986182,39986,39291,972.8,99.5,2.4,2944.5,6.2
